# Week 8 — Day 1: Healthcare Dataset Preparation

**Goal:** Build a clean instruction-tuning dataset (1,000+ samples) from real HuggingFace medical datasets covering:
- ✅ QA
- ✅ Reasoning
- ✅ Extraction

**Output:** `/data/train.jsonl`, `/data/val.jsonl`

## 0 — Install dependencies

In [ ]:
# Run once in Colab
!pip install -q datasets

## 1 — Setup: clone utils and create folders

In [ ]:
import os, sys, json

# ── Create the week-8 directory structure ──────────────────────────────────
for folder in ['data', 'utils', 'notebooks', 'adapters', 'quantized',
               'benchmarks', 'inference', 'deploy']:
    os.makedirs(folder, exist_ok=True)

print('Folders ready:', os.listdir('.'))

Folders ready: ['.config', 'quantized', 'benchmarks', 'utils', 'deploy', 'notebooks', 'data', 'inference', 'adapters', 'sample_data']


In [ ]:
data_cleaner_src = '''
import json, re, os
from collections import Counter

def estimate_tokens(text):
    return max(1, len(text) // 4)

def normalise(text):
    if not isinstance(text, str): return ""
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\\s+", " ", text).strip()
    text = text.replace("\u2019", "\'").replace("\u2018", "\'")
    text = text.replace("\u201c", \'"\').replace("\u201d", \'"\' )
    return text

def is_valid(sample, min_out=10, max_out=512, max_inst=256):
    instruction = sample.get("instruction", "")
    output      = sample.get("output", "")
    if not instruction or not output: return False, "empty_field"
    out_t  = estimate_tokens(output)
    inst_t = estimate_tokens(instruction)
    if out_t  < min_out:  return False, "output_too_short"
    if out_t  > max_out:  return False, "output_too_long"
    if inst_t > max_inst: return False, "instruction_too_long"
    if len(re.sub(r"[^a-zA-Z]", "", output)) < 20: return False, "output_not_text"
    return True, ""

def deduplicate(samples):
    seen, unique = set(), []
    for s in samples:
        key = (s["instruction"].lower(), s["output"].lower())
        if key not in seen:
            seen.add(key); unique.append(s)
    return unique

def clean_dataset(raw, min_out=10, max_out=512, max_inst=256):
    stats, cleaned = Counter(), []
    for s in raw:
        s = {"instruction": normalise(s.get("instruction","")),
             "input":       normalise(s.get("input","")),
             "output":      normalise(s.get("output",""))}
        ok, reason = is_valid(s, min_out, max_out, max_inst)
        if ok: cleaned.append(s); stats["passed"] += 1
        else:  stats[f"removed_{reason}"] += 1
    before = len(cleaned)
    cleaned = deduplicate(cleaned)
    stats["removed_duplicate"] = before - len(cleaned)
    return cleaned, dict(stats)

def split_dataset(samples, val_ratio=0.1, seed=42):
    import random; random.seed(seed)
    s = samples[:]; random.shuffle(s)
    i = int(len(s) * (1 - val_ratio))
    return s[:i], s[i:]

def save_jsonl(samples, path):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "w") as f:
        for s in samples: f.write(json.dumps(s) + "\\n")
    print(f"  Saved {len(samples):,} → {path}")

def token_length_stats(samples):
    import statistics
    lengths = sorted([estimate_tokens(s["output"]) for s in samples])
    n = len(lengths)
    return {"count": n, "mean": round(statistics.mean(lengths),1),
            "min": lengths[0], "max": lengths[-1],
            "p50": lengths[n//2], "p95": lengths[int(n*0.95)]}
'''

with open('utils/data_cleaner.py', 'w') as f:
    f.write(data_cleaner_src)
print('data_cleaner.py written to utils/')

data_cleaner.py written to utils/


## 2 — Load datasets from HuggingFace

| Type | Dataset | HF ID |
|------|---------|-------|
| QA | MedQuAD (via medalpaca) | `medalpaca/medical_meadow_medqa` |
| Reasoning | PubMedQA | `qiaojin/PubMedQA` |
| Extraction | MedMentions / BC5CDR-style | `medalpaca/medical_meadow_wikidoc` |

In [ ]:
from datasets import load_dataset

# ── TYPE 1: QA — Medical Q&A pairs ─────────────────────────────────────────
# medalpaca/medical_meadow_medqa: instruction + output format, clean medical QA
print('Loading QA dataset...')
ds_qa = load_dataset('medalpaca/medical_meadow_medqa', split='train')
print(f'  QA raw samples: {len(ds_qa):,}')
print(f'  Columns: {ds_qa.column_names}')
print(f'  Example:\n  {ds_qa[0]}')

Loading QA dataset...
  QA raw samples: 10,178
  Columns: ['input', 'instruction', 'output']
  Example:
  {'input': "Q:A 23-year-old pregnant woman at 22 weeks gestation presents with burning upon urination. She states it started 1 day ago and has been worsening despite drinking more water and taking cranberry extract. She otherwise feels well and is followed by a doctor for her pregnancy. Her temperature is 97.7°F (36.5°C), blood pressure is 122/77 mmHg, pulse is 80/min, respirations are 19/min, and oxygen saturation is 98% on room air. Physical exam is notable for an absence of costovertebral angle tenderness and a gravid uterus. Which of the following is the best treatment for this patient?? \n{'A': 'Ampicillin', 'B': 'Ceftriaxone', 'C': 'Ciprofloxacin', 'D': 'Doxycycline', 'E': 'Nitrofurantoin'},", 'instruction': 'Please answer with one of the option in the bracket', 'output': 'E: Nitrofurantoin'}


In [ ]:
# ── TYPE 2: Reasoning — PubMedQA ────────────────────────────────────────────
# pqa_labeled: 1k expert-labelled PubMed abstracts with yes/no/maybe + reasoning
print('Loading Reasoning dataset...')
ds_reason = load_dataset('qiaojin/PubMedQA', 'pqa_labeled', split='train')
print(f'  Reasoning raw samples: {len(ds_reason):,}')
print(f'  Columns: {ds_reason.column_names}')
print(f'  Example:\n  {ds_reason[0]}')

Loading Reasoning dataset...
  Reasoning raw samples: 1,000
  Columns: ['pubid', 'question', 'context', 'long_answer', 'final_decision']
  Example:
  {'pubid': 21645374, 'question': 'Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?', 'context': {'contexts': ['Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants.', 'The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole withi

In [ ]:
# ── TYPE 3: Extraction — WikiDoc Patient Info ───────────────────────────────
# wikidoc_patient_information: extract key medical facts from patient-facing text
print('Loading Extraction dataset...')
ds_extract = load_dataset('medalpaca/medical_meadow_wikidoc_patient_information', split='train')
print(f'  Extraction raw samples: {len(ds_extract):,}')
print(f'  Columns: {ds_extract.column_names}')
print(f'  Example:\n  {ds_extract[0]}')

Loading Extraction dataset...
  Extraction raw samples: 5,942
  Columns: ['input', 'output', 'instruction']
  Example:
  {'input': 'What are the symptoms of Allergy?', 'output': 'Allergy symptoms vary, but may include:\nBreathing problems (coughing, shortness of breath) Burning, tearing, or itchy eyes Conjunctivitis (red, swollen eyes) Coughing Diarrhea Headache Hives Itching of the nose, mouth, throat, skin, or any other area Runny nose Skin rashes Stomach cramps Vomiting Wheezing\nWhat part of the body is contacted by the allergen plays a role in the symptoms you develop. For example:\nAllergens that are breathed in often cause a stuffy nose, itchy nose and throat, mucus production, cough, or wheezing. Allergens that touch the eyes may cause itchy, watery, red, swollen eyes. Eating something you are allergic to can cause nausea, vomiting, abdominal pain, cramping, diarrhea, or a severe, life-threatening reaction. Allergens that touch the skin can cause a skin rash, hives, itching, bl

## 3 — Map each dataset to unified JSONL format

All samples become: `{"instruction": ..., "input": ..., "output": ...}`

Key design choices:
- **QA**: instruction = the question, input = "", output = answer
- **Reasoning**: instruction = reasoning task, input = abstract context, output = answer + long_answer explanation
- **Extraction**: instruction = extraction prompt, input = patient text, output = the wikidoc output

In [ ]:
# ── Map QA ──────────────────────────────────────────────────────────────────
# medalpaca datasets already have 'instruction', 'input', 'output' columns
qa_samples = []
for row in ds_qa:
    qa_samples.append({
        "instruction": row.get("instruction", ""),
        "input":       row.get("input", ""),
        "output":      row.get("output", "")
    })

print(f'QA samples mapped: {len(qa_samples):,}')

QA samples mapped: 10,178


In [ ]:
# ── Map Reasoning (PubMedQA) ────────────────────────────────────────────────
# PubMedQA structure: question, context (list of sentences), final_decision, long_answer
reason_samples = []
for row in ds_reason:
    # Combine context sentences into a paragraph
    context_sentences = row.get("context", {}).get("contexts", [])
    context_text = " ".join(context_sentences) if context_sentences else ""

    final_decision = row.get("final_decision", "")
    long_answer    = row.get("long_answer", "")

    # Output = decision + explanation (gives the model something meaningful to generate)
    output = f"Answer: {final_decision}. {long_answer}".strip()

    reason_samples.append({
        "instruction": "Based on the provided medical research context, answer the question and explain your reasoning.",
        "input":       f"Question: {row.get('question', '')}\n\nContext: {context_text}",
        "output":      output
    })

print(f'Reasoning samples mapped: {len(reason_samples):,}')

Reasoning samples mapped: 1,000


In [ ]:
# ── Map Extraction (WikiDoc) ─────────────────────────────────────────────────
# wikidoc_patient_information already has instruction / input / output format
extract_samples = []
for row in ds_extract:
    extract_samples.append({
        "instruction": row.get("instruction", ""),
        "input":       row.get("input", ""),
        "output":      row.get("output", "")
    })

print(f'Extraction samples mapped: {len(extract_samples):,}')
print(f'\nTotal before cleaning: {len(qa_samples)+len(reason_samples)+len(extract_samples):,}')

Extraction samples mapped: 5,942

Total before cleaning: 17,120


## 4 — Clean each split, then merge

The cleaner will:
1. Normalise text (strip HTML, fix encoding)
2. Remove samples with output too short (<10 tokens) or too long (>512 tokens)
3. Remove samples with instruction too long (>256 tokens)
4. Remove exact duplicates

In [ ]:
sys.path.insert(0, '.')
from utils.data_cleaner import clean_dataset, split_dataset, save_jsonl, token_length_stats

print('=== Cleaning QA ===')
qa_clean, qa_stats = clean_dataset(qa_samples,min_out=0)
print(f'  Stats: {qa_stats}')

print('\n=== Cleaning Reasoning ===')
reason_clean, reason_stats = clean_dataset(reason_samples)
print(f'  Stats: {reason_stats}')

print('\n=== Cleaning Extraction ===')
extract_clean, extract_stats = clean_dataset(extract_samples)
print(f'  Stats: {extract_stats}')

=== Cleaning QA ===


TypeError: clean_dataset() got an unexpected keyword argument 'skip_text_check'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Cap each type to keep a balanced dataset ────────────────────────────────
# We aim for ~1200 total: 500 QA + 500 Extraction + all Reasoning (~200)
import random
random.seed(42)

qa_final      = random.sample(qa_clean,      min(500, len(qa_clean)))
reason_final  = reason_clean   # small dataset, keep all
extract_final = random.sample(extract_clean, min(500, len(extract_clean)))

# Tag each sample with its type (useful for analysis, stripped before saving)
all_samples = []
for s in qa_final:      all_samples.append({**s, "_type": "qa"})
for s in reason_final:  all_samples.append({**s, "_type": "reasoning"})
for s in extract_final: all_samples.append({**s, "_type": "extraction"})

print(f'QA:        {len(qa_final):,}')
print(f'Reasoning: {len(reason_final):,}')
print(f'Extraction:{len(extract_final):,}')
print(f'Total:     {len(all_samples):,}')

## 5 — Token length analysis & distribution graph

In [ ]:
import matplotlib.pyplot as plt

def estimate_tokens(text): return max(1, len(text) // 4)

output_lengths = [estimate_tokens(s['output']) for s in all_samples]
input_lengths  = [estimate_tokens(s['input'])  for s in all_samples]
inst_lengths   = [estimate_tokens(s['instruction']) for s in all_samples]

# ── Distribution plot ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Healthcare Dataset — Token Length Distributions', fontsize=14, fontweight='bold')

for ax, lengths, label, color in zip(
    axes,
    [inst_lengths, input_lengths, output_lengths],
    ['Instruction', 'Input (context)', 'Output'],
    ['steelblue', 'seagreen', 'tomato']
):
    ax.hist(lengths, bins=40, color=color, edgecolor='white', alpha=0.85)
    ax.set_title(f'{label} Length')
    ax.set_xlabel('Estimated Tokens')
    ax.set_ylabel('Sample Count')
    mean_val = sum(lengths) / len(lengths)
    ax.axvline(mean_val, color='black', linestyle='--', linewidth=1.2, label=f'mean={mean_val:.0f}')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('data/token_distribution.png', dpi=120)
plt.show()
print('Plot saved to data/token_distribution.png')

In [ ]:
# ── Type distribution pie chart ──────────────────────────────────────────────
from collections import Counter
type_counts = Counter(s['_type'] for s in all_samples)

fig, ax = plt.subplots(figsize=(5, 5))
ax.pie(type_counts.values(), labels=type_counts.keys(),
       autopct='%1.1f%%', colors=['steelblue', 'seagreen', 'tomato'],
       startangle=140)
ax.set_title('Sample Type Distribution', fontweight='bold')
plt.savefig('data/type_distribution.png', dpi=120)
plt.show()

# ── Print summary stats ───────────────────────────────────────────────────────
print('\n── Output Token Stats ──────────────────')
for label, lengths in [('QA',        [estimate_tokens(s['output']) for s in qa_final]),
                        ('Reasoning', [estimate_tokens(s['output']) for s in reason_final]),
                        ('Extraction',[estimate_tokens(s['output']) for s in extract_final])]:
    if lengths:
        lengths.sort()
        print(f'{label:12s}: mean={sum(lengths)/len(lengths):.0f}  '
              f'min={lengths[0]}  max={lengths[-1]}  '
              f'p95={lengths[int(len(lengths)*0.95)]}')

## 6 — Save train.jsonl and val.jsonl

We remove the `_type` tag before saving (it was only for analysis).

In [ ]:
# Strip the _type tag before saving
clean_samples = [{k: v for k, v in s.items() if k != '_type'} for s in all_samples]

train_data, val_data = split_dataset(clean_samples, val_ratio=0.1, seed=42)

save_jsonl(train_data, 'data/train.jsonl')
save_jsonl(val_data,   'data/val.jsonl')

print(f'\nFinal split → Train: {len(train_data):,} | Val: {len(val_data):,}')

In [ ]:
# ── Sanity check: inspect a sample from each type ───────────────────────────
import json
print('── Sample: QA ──────────────────────────────────────────────')
print(json.dumps(qa_final[0], indent=2))

print('\n── Sample: Reasoning ───────────────────────────────────────')
print(json.dumps(reason_final[0], indent=2))

print('\n── Sample: Extraction ──────────────────────────────────────')
print(json.dumps(extract_final[0], indent=2))

## ✅ Day 1 Complete

| File | Description |
|------|-------------|
| `data/train.jsonl` | 90% of cleaned samples (~1,080+) |
| `data/val.jsonl` | 10% of cleaned samples (~120+) |
| `utils/data_cleaner.py` | Cleaning, dedup, split utilities |
| `data/token_distribution.png` | Token length histograms |
| `data/type_distribution.png` | QA / Reasoning / Extraction split |

Next: **Day 2 — LoRA / QLoRA Fine-Tuning**